# Load, apply, and compare all saved classifiers

This notebook compares the logistic regression, BDTs, compact top-particle MLP, PFN,
Particle Transformer, and ParticleNet-style network on exactly the same held-out jets.
It reconstructs each neural architecture from its JSON configuration and weights-only
checkpoint and loads common-test predictions saved by the baseline notebook. A **checkpoint**
stores learned parameter values so inference can be performed without training again.
**Inference** means applying a trained model to obtain scores.

Direct comparisons are allowed only on the same dataset and split fingerprint. Otherwise a
score difference might come from easier test examples rather than a better architecture.


## 0. Student controls

Use the same input sample as the training notebooks. Set `EVAL_PATH` to another compatible
Parquet file only for the optional final application section.


In [ ]:
# Student control: edit this value, then run the notebook from the top.
SOURCE_PATH = 'data/inclusive_jets.parquet'

# Optional independent sample to score after the common comparison.
EVAL_PATH = None


## 1. Environment


In [ ]:
import importlib.util
required = ['numpy', 'pandas', 'pyarrow', 'matplotlib', 'sklearn', 'torch', 'tqdm']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Missing packages: {missing}. From a terminal in this directory run "
        "./setup_student_env.sh (or use --current inside an existing henv), "
        "restart Jupyter from that henv, and select its registered kernel."
    )

import json, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm
import qg_constituent_ml as qg

DEVICE = qg.choose_device()
RUN_MODE_VALUE = globals().get('RUN_MODE')
if RUN_MODE_VALUE is not None and RUN_MODE_VALUE not in {'quick', 'full'}:
    raise ValueError("RUN_MODE must be 'quick' or 'full'")
SOURCE = Path(SOURCE_PATH)
print(f'PyTorch {torch.__version__}; built with CUDA {torch.version.cuda}')
print(f'device={DEVICE}' + (f'; GPU={torch.cuda.get_device_name(0)}' if DEVICE.type == 'cuda' else ''))
print((f'run mode={RUN_MODE_VALUE}; ' if RUN_MODE_VALUE else '') + f'source={SOURCE}')


In [ ]:
prepared=qg.prepare_dataset(SOURCE); manifest=qg.load_manifest(prepared)
bundles=qg.discover_bundles(dataset_fingerprint=manifest['source_sha256'])
if not bundles: raise FileNotFoundError('Train at least one architecture notebook first.')
print('\n'.join(map(str,bundles)))


## 2. Reload checkpoints and reproduce their predictions

Reproducing the saved scores is a consistency test: the architecture, preprocessing, and
weights were all restored correctly. The table reports parameter count and classification
metrics. A fair comparison uses identical test jets and never selects a winner by repeatedly
looking at the test set.


In [ ]:
from sklearn.metrics import roc_curve
rows=[]; curves={}; reference=None
for bundle in tqdm(bundles, desc='Evaluating saved models', unit='model'):
    model,config=qg.load_model_bundle(bundle,DEVICE)
    loaders=qg.make_loaders(prepared,config['architecture'],config['mode'])
    pred=qg.predict(model,loaders[2],DEVICE,progress=True,
                    description=f"Evaluating {config['architecture']}"); saved=np.load(bundle/'predictions.npz')
    assert np.array_equal(pred['event_ids'],saved['event_ids']) and np.array_equal(pred['jet_ids'],saved['jet_ids'])
    assert np.allclose(pred['scores'],saved['scores'],atol=2e-5)
    metrics=qg.binary_metrics(pred['labels'],pred['scores']); metrics['model']=f"{config['architecture']} ({config['mode']})"
    metrics['family']='all-constituent neural network'
    metrics['parameters']=sum(p.numel() for p in model.parameters()); metrics['bundle']=str(bundle); rows.append(metrics)
    curves[metrics['model']]=(*roc_curve(pred['labels'],pred['scores'])[:2],metrics['roc_auc'])
    current=(pred['event_ids'],pred['jet_ids'],pred['labels'])
    if reference is None: reference=current
    else:
        assert all(np.array_equal(a,b) for a,b in zip(reference,current))

baseline_bundle=Path('artifacts/qg_baselines')/manifest['source_sha256'][:12]
baseline_predictions=baseline_bundle/'predictions.npz'
if baseline_predictions.exists():
    import joblib
    baseline_config=json.loads((baseline_bundle/'config.json').read_text())
    baseline_models=joblib.load(baseline_bundle/'models.joblib')
    assert baseline_config['dataset_fingerprint']==manifest['source_sha256']
    assert baseline_config['split_fingerprint']==manifest['split_fingerprint']
    assert set(baseline_models['models'])==set(baseline_config['model_names'])
    saved=np.load(baseline_predictions)
    current=(saved['event_ids'],saved['jet_ids'],saved['labels'])
    if reference is not None:
        assert all(np.array_equal(a,b) for a,b in zip(reference,current)), 'Baseline and neural test jets differ'
    for name,score in zip(saved['model_names'].astype(str),saved['scores']):
        metrics=qg.binary_metrics(saved['labels'],score); metrics['model']=name
        metrics['family']='classical / compact baseline'; metrics['parameters']=np.nan
        metrics['bundle']=str(baseline_bundle); rows.append(metrics)
        curves[name]=(*roc_curve(saved['labels'],score)[:2],metrics['roc_auc'])
else:
    print(f'Baseline bundle not found at {baseline_bundle}. Run the classification notebook to add logistic, BDT, and MLP results.')

import pandas as pd
results=pd.DataFrame(rows).sort_values('roc_auc',ascending=False); display(results)


In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
for name,(fpr,tpr,auc) in curves.items(): ax.plot(tpr,1/np.clip(fpr,1e-3,None),label=f'{name}: {auc:.3f}')
ax.set(xlabel='quark efficiency',ylabel='gluon rejection',yscale='log',title='All models on the same held-out jets')
ax.legend(); plt.tight_layout(); plt.show()


### How to read this figure

Every curve uses the **same held-out test sample**: jets set aside before training whose
labels did not adjust model weights, choose settings, or trigger early stopping. Therefore
differences are attributable to the trained models rather than to easier or harder test
examples. Moving right keeps more true quark jets
(quark efficiency). Moving up rejects more gluon jets; rejection 20 means that only about
$1/20=5\%$ of gluon jets pass. Because the vertical axis is logarithmic, a change from 5 to
10 is as large multiplicatively as a change from 10 to 20.

Look first at the efficiency region relevant to the intended analysis: curves can cross, so
the model with the largest AUC need not be best at every operating point. The AUC in each
legend entry summarizes the full curve (0.5 random, 1.0 perfect). Small separations should not
be over-interpreted without statistical uncertainties or another independent test sample.


## 3. Apply a model to another sample

Set `EVAL_PATH` in the student-controls cell to evaluate a different compatible Parquet file. The
saved training normalization is reused; it must never be refitted on the evaluation sample.
Refitting would allow the new sample to alter preprocessing and would make its scores
inconsistent with the original model.


In [ ]:
if EVAL_PATH:
    evaluation,config=qg.predict_parquet(bundles[0],EVAL_PATH,device=DEVICE)
    labeled=evaluation['labels'] >= 0
    print(f"Scored all {len(evaluation['scores']):,} jets from {EVAL_PATH}; {labeled.sum():,} have quark/gluon labels")
    if labeled.any(): print(json.dumps(qg.binary_metrics(evaluation['labels'][labeled],evaluation['scores'][labeled]),indent=2))
else: print('EVAL_PATH is unset; common-test comparison complete.')
